# DeepSeek-V2-Lite -> W4A16 (GPTQ, router fp16) — one-time quantization

Self-quantizes the routed experts to 4-bit and keeps the router in fp16, then pushes to
`Ryze242005/DeepSeek-V2-Lite-w4a16-gptq` (private). See scripts/quantize_deepseek.py and configs/models.yaml (deepseek vllm block).

**Before running:** GPU **T4 x2**, **Internet On**, **Add Input -> the mixed-v2 corpus dataset**
(calibration), and Kaggle **Secrets**: `HF_TOKEN` (write) + `GITHUB_TOKEN`. One-time; paste back the
`verify OK` + `pushed to ...` lines.


In [ ]:
# Cell 1 — runtime audit (subprocess; does not import torch into the kernel).
import subprocess
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,compute_cap",
     "--format=csv"], capture_output=True, text=True).stdout, flush=True)
print("EXPECT two rows, compute_cap 7.5.")


In [ ]:
# Cell 2 — install GPTQModel + deps. Do NOT restart the kernel after.
import subprocess, sys


def sh(args):
    print("$", " ".join(args), flush=True)
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "")[-3000:], flush=True)
    print("exit:", p.returncode, flush=True)
    return p.returncode


sh([sys.executable, "-m", "pip", "install", "-q", "-U", "gptqmodel", "--no-build-isolation"])
sh([sys.executable, "-m", "pip", "install", "-q", "transformers==4.55.2", "accelerate", "datasets"])
chk = subprocess.run(
    [sys.executable, "-c", "import gptqmodel, torch; print('IMPORT_OK', gptqmodel.__version__)"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(chk.stdout, flush=True)
print(">>> Proceed only if IMPORT_OK printed. DO NOT restart the kernel.")


In [ ]:
# Cell 3 — clone the repo, put it on sys.path + PYTHONPATH.
import os, subprocess, sys
from pathlib import Path

GIT_URL = "https://github.com/ryzewtf/GenAI-IA-1.git"
GIT_REF = "VLLM_PORT"
REPO = Path("/kaggle/working/repo")


def run(cmd, cwd=None, check=True, quiet=False):
    if not quiet:
        print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd), text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout and not quiet:
        print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit(f"FAILED ({p.returncode}): {' '.join(str(c) for c in cmd)}")
    return p


url = GIT_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = GIT_URL.replace("https://", f"https://{tok}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning anonymously")

if REPO.exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO)
    run(["git", "checkout", GIT_REF], cwd=REPO)
    run(["git", "pull", "--ff-only"], cwd=REPO, check=False)
else:
    run(["git", "clone", url, str(REPO)])
    run(["git", "checkout", GIT_REF], cwd=REPO)
print("repo at", run(["git", "rev-parse", "HEAD"], cwd=REPO, quiet=True).stdout.strip())

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO)


In [ ]:
# Cell 4 — locate the mixed-v2 corpus (calibration) and export HF_TOKEN.
import os
from pathlib import Path

CORPUS = Path("/kaggle/input/moe-corpus-v2/mixed-v2.jsonl")
assert CORPUS.exists(), f"{CORPUS} not found — Add Input -> the mixed-v2 dataset"
print("calib corpus:", CORPUS, f"({CORPUS.stat().st_size/1e6:.1f} MB)")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("WARNING: no HF_TOKEN secret -", e, "(the push will fail without it)")


In [ ]:
# Cell 5 — quantize (router fp16) + verify + push to Ryze242005/DeepSeek-V2-Lite-w4a16-gptq (private).
import subprocess, sys

argv = [
    sys.executable, "-m", "scripts.quantize_deepseek",
    "--model-id", "deepseek-ai/DeepSeek-V2-Lite",
    "--out", "/tmp/deepseek-w4a16",
    "--repo-id", "Ryze242005/DeepSeek-V2-Lite-w4a16-gptq",
    "--calib-corpus", str(CORPUS),
    "--calib-samples", "256",
    "--push",
]
print("$", " ".join(argv), flush=True)
rc = subprocess.run(argv, text=True).returncode
print(f"\n== quantize exit {rc}", flush=True)
print("OK — quantized model pushed" if rc == 0 else "FAILED (see above)")
